In [1]:
import pandas as pd
import requests
import time
import re

In [2]:
# Setup
API_KEY = "AHFKINXhLc4MAajui0VAQ8wtBbf5iHHq79weice5"
BASE_URL = "https://commonchemistry.cas.org/api"
data_path = "data/CDPH Search results - 1_12_2026, 3_34_15 PM.csv"
data = pd.read_csv(data_path, low_memory = False)

# Function to extract CAS numbers that are already in the ingredient name
def get_cas_from_text(text):
    if not text.strip():
        return ""
    cas_numbers = re.findall(r'\b\d{2,7}-\d{2}-\d\b', str(text))
    return cas_numbers if cas_numbers else ""

# Function to clean ingredient name (remove CAS numbers)
def clean_name(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\d{2,7}-\d{2}-\d', '', text)  # Remove CAS
    text = re.sub(r'/\s*\d+[-\d/\s]*', '', text)   # Remove extra numbers
    return text.strip()


def get_official_name(unique_cas):
    if not unique_cas or unique_cas == "":
        return ""
    url = f"{BASE_URL}/detail"
    headers = {"X-Api-Key": API_KEY}
    params = {"cas_rn": unique_cas}
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            return response.json().get('name', 'NOT FOUND')
        else:
            return "NOT FOUND"
    except:
        return "ERROR"

In [3]:
# Setup
API_KEY = "AHFKINXhLc4MAajui0VAQ8wtBbf5iHHq79weice5"
BASE_URL = "https://commonchemistry.cas.org/api"
data_path = "data/CDPH Search results - 1_12_2026, 3_34_15 PM.csv"
data = pd.read_csv(data_path, low_memory = False)

# Function to extract CAS numbers that are already in the ingredient name
def get_cas_from_text(text):
    if not text.strip():
        return ""
    cas_numbers = re.findall(r'\b\d{2,7}-\d{2}-\d\b', str(text))
    return cas_numbers if cas_numbers else ""

# Function to clean ingredient name (remove CAS numbers)
def clean_name(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\d{2,7}-\d{2}-\d', '', text)  # Remove CAS
    text = re.sub(r'/\s*\d+[-\d/\s]*', '', text)   # Remove extra numbers
    text = re.sub(r'\s*/\s*', ' ', text)              # Replace / with spaces
    text = re.sub(r'\s+', ' ', text) 
    return text.strip()


def get_official_name(unique_cas):
    if not unique_cas or unique_cas == "":
        return ""
    url = f"{BASE_URL}/detail"
    headers = {"X-Api-Key": API_KEY}
    params = {"cas_rn": unique_cas}
    try:
        response = requests.get(url, headers=headers, params=params)
        if response.status_code == 200:
            return response.json().get('name', 'NOT FOUND')
        else:
            return "NOT FOUND"
    except:
        return "ERROR"

In [4]:
#Find CAS numbers
unique_cas = data["Ingredient Name"].unique()
#pd.DataFrame(unique_cas, columns = ["Ingredient Name"]).to_csv("ingredients.csv", index = False)

In [5]:
errors = []
cas_metadata = {}
for ingredient in unique_cas:
        cas_metadata[ingredient] = {}

In [6]:
cas_metadata

{'Titanium dioxide (CI 77891) 13463-67-7 / 1317-70-0 / 1317-80-2 / 98084-96-9': {},
 'CITRONELLOL ': {},
 'LINALOOL ': {},
 'HEXYL CINNAMAL ': {},
 'Limonene (1-methyl-4-prop-1-en-2-yl-cyclohexene; dl-limonene (racemic); Dipentene; (R)-p-mentha-1,8-diene; (d-limonene); (S)-p-mentha-1,8-diene; (l-limonene)) 5989-27-5 / 138-86-3 / 7705-14-8 / 5989-54-8': {},
 'Lilial ': {},
 'Aloe vera, non-decolorized (unfiltered) whole leaf extract ': {},
 '1,6-Octadien-3-ol, 3,7-dimethyl- (LINALOOL) 78-70-6': {},
 'CITRAL ': {},
 'GERANIOL ': {},
 'Citronellol /+- 3,7-dimethyloct-6-en-1-ol (CITRONELLOL; (3R)-3,7-dimethyloct-6-en-1-ol; (3S)-3,7-dimethyloct-6-en-1-ol) 106-22-9 / 26489-01-0 / 1117-61-9 / 7540-51-4': {},
 'Benzyl benzoate 120-51-4': {},
 'BUTYLPHENYL METHYLPROPIONAL ': {},
 '2-Benzylideneoctanal (HEXYL CINNAMAL) 101-86-0': {},
 '2-Phenoxyethyl isobutyrate 103-60-6': {},
 '1,3,4,6,7,8-Hexahydro-4,6,6,7,8,8-hexamethylcyclopenta-?-2-benzopyran (Hexamethylindanopyran) 1222-05-5': {},
 'Isoeug

In [7]:
# GO THROUGH DICTIONARY and apply ur functions
for ingredient, empty_dict in cas_metadata.items():
    try:
        cas_number = get_cas_from_text(ingredient)
    except AttributeError:
        cas_number = ""
    clean_name_str = clean_name(ingredient)
    empty_dict.update({
        'cas_number': cas_number,
        'clean_name': clean_name_str,
    })


In [8]:
cas_data_to_write = []
for ingredient, data in cas_metadata.items():
    cas_data_to_write.append([
        ingredient,
        data['cas_number'],
        data['clean_name']
    ])
import csv
with open('data/ingredients_cleaned.csv', 'w') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['original_ingredient', 'cas_number', 'clean_name'])
    writer.writerows(cas_data_to_write)

In [62]:
#test by CAS number
test_cas = ['80-54-6']

url = f"{BASE_URL}/detail"
headers = {"X-Api-Key": API_KEY}
params = {"cas_rn": test_cas}

response = requests.get(url, headers=headers, params=params)

#temp_data = []
#for cas_num in test_cas:
    #response = requests.get(url, headers=headers, params={'cas_rn': cas_num})
    #temp_data.append(response.json())


name = response.json().get('name')
print(name)

#print(f"Status Code: {response.status_code}")
#print(f"\nResponse JSON:")
#print(response.json())

Lilial


In [61]:
response.json()

{'uri': 'substance/pt/13463677',
 'rn': '13463-67-7',
 'name': 'Titania',
 'images': ['<svg width="100" viewBox="0 0 100 15" style="fill-opacity:1; color-rendering:auto; color-interpolation:auto; text-rendering:auto; stroke:black; stroke-linecap:square; stroke-miterlimit:10; shape-rendering:auto; stroke-opacity:1; fill:black; stroke-dasharray:none; font-weight:normal; stroke-width:1; font-family:\'Open Sans\'; font-style:normal; stroke-linejoin:miter; font-size:12; stroke-dashoffset:0; image-rendering:auto;" height="15" class="cas-substance-image" xmlns:xlink="http://www.w3.org/1999/xlink" xmlns="http://www.w3.org/2000/svg"><svg class="cas-substance-single-component"><rect y="0" x="0" width="100" stroke="none" ry="7" rx="7" height="15" fill="white" class="cas-substance-group"/><svg y="0" x="13" width="74" viewBox="0 0 74 15" style="fill:black;" height="15" class="cas-substance-single-component-image"><svg><g><g transform="translate(37,8)" style="text-rendering:geometricPrecision; color

In [59]:
[row.get('name') for row in temp_data]

['Titania', 'Anatase (TiO<sub>2</sub>)', 'Rutile (TiO<sub>2</sub>)', 'Titania']

In [56]:
response.request.url

'https://commonchemistry.cas.org/api/detail?cas_rn=13463-67-7&cas_rn=1317-70-0&cas_rn=1317-80-2&cas_rn=98084-96-9'

In [38]:
#test by Name
test_name = "Styrene"

url = f"{BASE_URL}/search"
headers = {"X-Api-Key": API_KEY}
params = {"q": test_name}

response = requests.get(url, headers=headers, params=params)

if response.status_code == 200:
    data = response.json()
    results = data.get('results', [])
    
    if results:
        # Get the first result
        rn = results[0].get('rn')
        print(rn)
    else:
        print("No results found")
else:
    print(f"Error: {response.status_code}")

100-42-5


In [ ]:
counter = 0
for ingredient, metadata in cas_metadata.items():
    
    if counter >= 10:
        break
    
    cas_num = metadata['cas_number']
    clean_name_str = metadata['clean_name']

    if cas_num and cas_num != "":
        if isinstance(cas_num, list):
            cas_num = cas_num[0]

        url = f"{BASE_URL}/detail"
        headers = {"X-Api-Key": API_KEY}
        params = {"cas_rn": cas_num}
        response = requests.get(url, headers=headers, params=params)  

        if response.status_code == 200:  # Fixed indentation here
            official_name = response.json().get('name', '')
            metadata['official_name'] = official_name
            metadata['lookup_type'] = 'CAS'
        else:
            metadata['official_name'] = 'NOT FOUND'
            metadata['lookup_type'] = 'CAS' 
        
    elif clean_name_str and clean_name_str != "":
        url = f"{BASE_URL}/search"
        headers = {"X-Api-Key": API_KEY}
        params = {"q": clean_name_str}
        response = requests.get(url, headers=headers, params=params)

        if response.status_code == 200:
            results = response.json()
            if results.get('count', 0) > 0:
                official_name = results['results'][0].get('name', '')
                metadata['official_name'] = official_name
                metadata['lookup_type'] = 'NAME'
            else:
                metadata['official_name'] = 'NOT FOUND'
                metadata['lookup_type'] = 'NAME'
        else:
            metadata['official_name'] = 'NOT FOUND'
            metadata['lookup_type'] = 'NAME'
    
    else:
        metadata['official_name'] = 'NO DATA'
        metadata['lookup_type'] = 'NONE'

    print(f"Done {counter + 1}/10: {ingredient[:30]}")
    counter += 1

Done 1/10: Titanium dioxide (CI 77891) 13
Done 2/10: CITRONELLOL 
Done 3/10: LINALOOL 
Done 4/10: HEXYL CINNAMAL 
Done 5/10: Limonene (1-methyl-4-prop-1-en
Done 6/10: Lilial 
Done 7/10: Aloe vera, non-decolorized (un
Done 8/10: 1,6-Octadien-3-ol, 3,7-dimethy
Done 9/10: CITRAL 
Done 10/10: GERANIOL 


In [26]:
# API call
counter = 0
for ingredient, data in cas_metadata.items():

    if counter >= 5:  # Stop after 5 iterations
        break

    cas_num = data['cas_number']
    clean_name_str = data['clean_name']

    if cas_num and cas_num != "":
        if isinstance(cas_num, list):
            cas_num = cas_num[0]

        url = f"{BASE_URL}/detail"
        headers = {"X-Api-Key": API_KEY}
        params = {"cas_rn": cas_num}
        response = requests.get(url, headers=headers, params=params)  

        if response.status_code == 200:
            official_name = response.json().get('name', '')
            metadata['official_name'] = official_name
            metadata['lookup_type'] = 'CAS'
        else:
            metadata['official_name'] = 'NOT FOUND'
            metadata['lookup_type'] = 'CAS' 
        
    elif clean_name_str and clean_name_str != "":
        url = f"{BASE_URL}/search"
        headers = {"X-Api-Key": API_KEY}
        params = {"q": clean_name_str}
        response = requests.get(url, headers=headers, params=params)

        if response.status_code == 200:
            results = response.json()
            if results.get('count', 0) > 0:
                official_name = results['results'][0].get('name', '')
                metadata['official_name'] = official_name
                metadata['lookup_type'] = 'NAME'
            else:
                metadata['official_name'] = 'NOT FOUND'
                metadata['lookup_type'] = 'NAME'
        else:
            metadata['official_name'] = 'NOT FOUND'
            metadata['lookup_type'] = 'NAME'
    
    else:
        metadata['official_name'] = 'NO DATA'
        metadata['lookup_type'] = 'NONE'

    print(f"Done {counter + 1}/5: {ingredient[:30]}")
    counter += 1  # Increment counter



Done 1/5: Titanium dioxide (CI 77891) 13
Done 2/5: CITRONELLOL 
Done 3/5: LINALOOL 
Done 4/5: HEXYL CINNAMAL 
Done 5/5: Limonene (1-methyl-4-prop-1-en


In [68]:
import csv
import requests
import time

# Your API key
API_KEY = "AHFKINXhLc4MAajui0VAQ8wtBbf5iHHq79weice5"

# Read the CSV and store results
results = []

with open('data/ingredients_cleaned.csv', 'r') as file:
    reader = csv.DictReader(file)
    
    for row in reader:
        ingredient = row['original_ingredient']
        cas_number = row['cas_number']
        clean_name = row['clean_name']
        if cas_number == "['80-54-6']":
            print("HELLO THERE")
        else:
            continue
        
        # Start with empty values
        api_rn = ''
        api_name = ''
        
        # Try to look up by CAS number first
        if cas_number:
            url = f"https://commonchemistry.cas.org/api/detail?cas_rn={cas_number}"
            headers = {"X-Api-Key": API_KEY}
            response = requests.get(url, headers=headers)
            
            if response.status_code >= 400:
                data = response.json()
                api_rn = data.get('rn', '')
                api_name = data.get('name', '')
        
        # If we didn't find it, try by clean name
        if not api_name and clean_name:
            url = f"https://commonchemistry.cas.org/api/search?q={clean_name}"
            headers = {"X-Api-Key": API_KEY}
            response = requests.get(url, headers=headers)
            
            if response.status_code == 200:
                data = response.json()
                if data.get('count', 0) > 0:
                    api_rn = data['results'][0].get('rn', '')
                    api_name = data['results'][0].get('name', '')
        
        # Save the result
        results.append([ingredient, cas_number, clean_name, api_rn, api_name])
        
        print(f"Done: {ingredient}")
        time.sleep(0.1)

# Write to new CSV
with open('data/ingredients_with_api_data.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['original_ingredient', 'cas_number', 'clean_name', 'api_rn', 'api_name'])
    writer.writerows(results)

print("All done!")

HELLO THERE
Done: 2-(4-tert-Butylbenzyl)propionaldehyde (Lilial; BUTYLPHENYL METHYLPROPIONAL) 80-54-6
All done!


In [70]:
import csv
import requests
import time

# Your API key
API_KEY = "AHFKINXhLc4MAajui0VAQ8wtBbf5iHHq79weice5"
BASE_URL = "https://commonchemistry.cas.org/api"

# Read the CSV and store results
results = []

with open('data/ingredients_cleaned.csv', 'r') as file:
    reader = csv.DictReader(file)
    
    for row in reader:
        ingredient = row['original_ingredient']
        
        # Safely evaluate cas_number from string representation
        cas_raw = row['cas_number'].strip()
        cas_number = None
        
        if cas_raw and cas_raw != '[]':  # Not empty and not empty list
            try:
                cas_list = eval(cas_raw)  # Convert string "[1, 2, 3]" to list
                if isinstance(cas_list, list) and len(cas_list) > 0:
                    cas_number = cas_list[0]  # Take first CAS number
                elif isinstance(cas_list, str):
                    cas_number = cas_list
            except:
                pass  # Skip if eval fails
        
        clean_name = row['clean_name']
        
        # Start with empty values
        api_rn = ''
        api_name = ''
        
        # Try to look up by CAS number first
        if cas_number:
            url = f"{BASE_URL}/detail"
            headers = {"X-Api-Key": API_KEY}
            params = {"cas_rn": cas_number}
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                api_rn = data.get('rn', '')
                api_name = data.get('name', '')
        
        # If we didn't find it, try by clean name
        if not api_name and clean_name:
            url = f"{BASE_URL}/search"
            headers = {"X-Api-Key": API_KEY}
            params = {"q": clean_name}
            response = requests.get(url, headers=headers, params=params)
            
            if response.status_code == 200:
                data = response.json()
                if data.get('count', 0) > 0:
                    api_rn = data['results'][0].get('rn', '')
                    api_name = data['results'][0].get('name', '')
        
        # Save the result
        results.append([ingredient, cas_number if cas_number else '', clean_name, api_rn, api_name])
        
        print(f"Done: {ingredient}")
        time.sleep(0.1)

# Write to new CSV
with open('data/ingredients_with_api_data.csv', 'w', newline='') as file:
    writer = csv.writer(file)
    writer.writerow(['original_ingredient', 'cas_number', 'clean_name', 'api_rn', 'api_name'])
    writer.writerows(results)

print("All done!")

Done: Titanium dioxide (CI 77891) 13463-67-7 / 1317-70-0 / 1317-80-2 / 98084-96-9
Done: CITRONELLOL 
Done: LINALOOL 
Done: HEXYL CINNAMAL 
Done: Limonene (1-methyl-4-prop-1-en-2-yl-cyclohexene; dl-limonene (racemic); Dipentene; (R)-p-mentha-1,8-diene; (d-limonene); (S)-p-mentha-1,8-diene; (l-limonene)) 5989-27-5 / 138-86-3 / 7705-14-8 / 5989-54-8
Done: Lilial 
Done: Aloe vera, non-decolorized (unfiltered) whole leaf extract 
Done: 1,6-Octadien-3-ol, 3,7-dimethyl- (LINALOOL) 78-70-6
Done: CITRAL 
Done: GERANIOL 
Done: Citronellol /+- 3,7-dimethyloct-6-en-1-ol (CITRONELLOL; (3R)-3,7-dimethyloct-6-en-1-ol; (3S)-3,7-dimethyloct-6-en-1-ol) 106-22-9 / 26489-01-0 / 1117-61-9 / 7540-51-4
Done: Benzyl benzoate 120-51-4
Done: BUTYLPHENYL METHYLPROPIONAL 
Done: 2-Benzylideneoctanal (HEXYL CINNAMAL) 101-86-0
Done: 2-Phenoxyethyl isobutyrate 103-60-6
Done: 1,3,4,6,7,8-Hexahydro-4,6,6,7,8,8-hexamethylcyclopenta-?-2-benzopyran (Hexamethylindanopyran) 1222-05-5
Done: Isoeugenol (Phenol, 2-methoxy-4-(1

In [33]:
final_data = []
for ingredient, metadata in cas_metadata.items():
    final_data.append([
        ingredient,
        metadata['cas_number'],
        metadata['clean_name'],
        metadata.get('official_name', ''),
        metadata.get('lookup_type', '')
    ])

with open('data/ingredients_final.csv', 'w', newline='') as outfile:
    writer = csv.writer(outfile)
    writer.writerow(['original_ingredient', 'cas_number', 'clean_name', 'official_name', 'lookup_type'])
    writer.writerows(final_data)


# Step Extract CAS numbers and clean names
print("Step 1: Processing ingredient names...")
unique_cas['CAS_Number'] = unique_cas['Ingredient Name'].apply(get_cas_from_text)
unique_cas['Clean_Name'] = unique_cas['Ingredient Name'].apply(clean_name)

# Step Save results
output_file = 'data/CDPH_ingredients_standardized.csv'
unique_cas.to_csv(output_file, index=False)

print(f"\n✓ Done! Results saved to: {output_file}")
print(f"\nSummary:")
print(f"Total rows: {len(unique_cas)}")
print(f"Rows with CAS numbers: {(unique_cas['CAS_Number'] != '').sum()}")
print(f"Official names retrieved: {(unique_cas['Official_Name'] != '').sum()}")

# Step 2: Get official names from API (only for rows that have CAS numbers)
print("\nStep 2: Getting official names from API...")
data['Official_Name'] = ""

rows_with_cas = data[data['CAS_Number'] != '']
total = len(rows_with_cas)

for i, (index, row) in enumerate(rows_with_cas.iterrows(), 1):
    print(f"[{i}/{total}] Getting name for CAS: {row['CAS_Number']}")
    official_name = get_official_name(row['CAS_Number'])
    data.at[index, 'Official_Name'] = official_name
    time.sleep(0.5)  # Be nice to the API